# 📘 智能体架构 6：规划器 → 执行器 → 验证器 (PEV)

在本笔记本中，我们将探索 **规划器 → 执行器 → 验证器 (PEV)** 架构，这是一种为智能体系统引入关键健壮性和自我纠正层的模式。这一架构的灵感来自严格的软件工程和质量保证流程，其中工作在经过验证之前不被视为"完成"。

虽然标准规划智能体带来了结构和可预测性，但它基于一个关键假设：其工具将每次都完美工作并返回有效数据。在现实世界中，API 会失败、搜索不返回结果、数据可能格式错误。PEV 模式通过添加一个专用的**验证器**智能体来解决此问题，该智能体在每个操作后充当质量保证检查，使系统能够检测失败并动态恢复。

### 定义
**规划器 → 执行器 → 验证器 (PEV)** 架构是一种三阶段工作流程，明确分离了规划、执行和验证行为。它确保每个步骤的输出在智能体继续之前得到验证，创建一个健壮的、自我纠正的循环。

### 高层工作流程

1. **规划：** "规划器"智能体将高层目标分解为一系列具体的、可执行的步骤。
2. **执行：** "执行器"智能体从计划中获取*下一步*并调用相应的工具。
3. **验证：** "验证器"智能体检查来自执行器的输出。它检查正确性、相关性和潜在错误。然后它产生判断：步骤是成功还是失败？
4. **路由和迭代：** 基于验证器的判断，路由器决定下一步行动：
   * 如果步骤**成功**且计划未完成，循环回执行器执行下一步。
   * 如果步骤**失败**，循环回规划器创建*新*计划，通常提供失败上下文以使新计划更智能。
   * 如果步骤**成功**且计划完成，进入最终综合步骤。

### 适用场景 / 应用
* **安全关键应用（金融、医疗保健）：** 当错误的成本很高时，PEV 提供必要的护栏，防止智能体基于错误数据采取行动。
* **具有不可靠工具的系统：** 当处理可能不稳定或返回不一致数据的外部 API 时，验证器可以优雅地捕获失败。
* **高精度任务（法律、科学）：** 对于需要高度事实准确性的任务，验证器确保每条检索的信息在用于下游推理之前是有效的。

### 优缺点
* **优点：**
    * **健壮性和可靠性：** 其核心优势是能够检测并从错误中恢复。
    * **模块化：** 关注点的分离使系统更易于调试和维护。
* **缺点：**
    * **增加的延迟和成本：** 在每个操作后添加验证步骤会增加更多 LLM 调用，使其成为我们迄今为止介绍的最慢和最昂贵的架构。
    * **验证器复杂性：** 设计有效的验证器可能具有挑战性。它需要足够智能以区分小问题和关键失败。

## 阶段 0：基础与环境设置

我们将从安装库和为 Anthropic、LangSmith 和我们的工具配置 API 密钥开始。

### 步骤 0.1：安装核心库

**我们要做什么：**
我们将安装本项目系列的标准库套件。

In [ ]:
# !pip install -q -U langchain-anthropic langchain langgraph rich python-dotenv langchain-tavily

### 步骤 0.2：导入库和设置密钥

**我们要做什么：**
我们将导入必要的模块并从 `.env` 文件加载我们的 API 密钥。

**需要的操作：** 在此目录中创建一个包含密钥的 `.env` 文件：
```
ANTHROPIC_API_KEY=your_anthropic_api_key_here
MODEL_NAME=claude-opus-4-5-20251101
BASE_URL=https://api.anthropic.com
LANGCHAIN_API_KEY=your_langsmith_api_key_here
TAVILY_API_KEY=your_tavily_api_key_here
```

In [ ]:
import os
import re
from typing import List, Annotated, TypedDict, Optional
from dotenv import load_dotenv
import json

# LangChain components
from langchain_anthropic import ChatAnthropic
from langchain_tavily import TavilySearch
from langchain_core.messages import BaseMessage, ToolMessage
from pydantic import BaseModel, Field

# LangGraph components
from langgraph.graph import StateGraph, END

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown

# --- API Key and Tracing Setup ---
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - PEV"

for key in ["ANTHROPIC_API_KEY", "LANGCHAIN_API_KEY", "TAVILY_API_KEY"]:
    if not os.environ.get(key):
        print(f"{key} not found. Please create a .env file and set it.")

print("Environment variables loaded and tracing is set up.")

## 阶段 1：基线 - 规划器-执行器智能体

为了理解验证器的必要性，我们必须首先构建一个没有验证器的智能体。这个智能体将创建一个计划并盲目地遵循它，演示当工具调用出错时失败的可能性。

### 步骤 1.1：构建规划器-执行器智能体

**我们要做什么：**
我们将构建一个简单的规划器-执行器图，类似于之前笔记本中的图。为了模拟现实世界的失败，我们将创建一个特殊的"不稳定"工具。这个工具将对特定查询故意返回错误消息，我们的基本智能体将无法处理。

In [ ]:
console = Console()
model = os.environ.get("MODEL_NAME", "claude-opus-4-5-20251101")
base_url = os.environ.get("BASE_URL")
llm = ChatAnthropic(model=model, base_url=base_url, temperature=0)

# Define a 'flaky' tool that will fail for a specific query
def flaky_web_search(query: str) -> str:
    """Performs a web search, but is designed to fail for a specific query."""
    console.print(f"--- TOOL: Searching for '{query}'... ---")
    if "employee count" in query.lower():
        console.print("--- TOOL: [bold red]Simulating API failure![/bold red] ---")
        return "Error: Could not retrieve data. The API endpoint is currently unavailable."
    else:
        result = TavilySearch(max_results=2).invoke(query)
        # 🔑 Ensure result is always a string
        if isinstance(result, (dict, list)):
            return json.dumps(result, indent=2)
        return str(result)

# Define the state for the basic P-E agent
class BasicPEState(TypedDict):
    user_request: str
    plan: Optional[List[str]]
    intermediate_steps: List[str]
    final_answer: Optional[str]

class Plan(BaseModel):
    steps: List[str] = Field(description="A list of tool calls to execute.")

def basic_planner_node(state: BasicPEState):
    console.print("--- (Basic) PLANNER: Creating plan... ---")
    planner_llm = llm.with_structured_output(Plan)

    prompt = f"""
    You are a planning agent. 
    Your job is to decompose the user's request into a list of clear tool queries.

    - Only return JSON that matches this schema: {{ "steps": [ "query1", "query2", ... ] }}
    - Do NOT return any prose or explanation.
    - Always use the 'flaky_web_search' tool for queries.

    User's request: "{state['user_request']}"
    """
    plan = planner_llm.invoke(prompt)
    return {"plan": plan.steps}

def basic_executor_node(state: BasicPEState):
    console.print("--- (Basic) EXECUTOR: Running next step... ---")
    next_step = state["plan"][0]
    result = flaky_web_search(next_step)
    return {"plan": state["plan"][1:], "intermediate_steps": state["intermediate_steps"] + [result]}

def basic_synthesizer_node(state: BasicPEState):
    console.print("--- (Basic) SYNTHESIZER: Generating final answer... ---")
    context = "\n".join(state["intermediate_steps"])
    prompt = f"Synthesize an answer for '{state['user_request']}' using this data:\n{context}"
    answer = llm.invoke(prompt).content
    return {"final_answer": answer}

# Build the graph
pe_graph_builder = StateGraph(BasicPEState)
pe_graph_builder.add_node("plan", basic_planner_node)
pe_graph_builder.add_node("execute", basic_executor_node)
pe_graph_builder.add_node("synthesize", basic_synthesizer_node)

pe_graph_builder.set_entry_point("plan")
pe_graph_builder.add_conditional_edges("plan", lambda s: "execute" if s["plan"] else "synthesize")
pe_graph_builder.add_conditional_edges("execute", lambda s: "execute" if s["plan"] else "synthesize")
pe_graph_builder.add_edge("synthesize", END)

basic_pe_app = pe_graph_builder.compile()
print("Basic Planner-Executor agent compiled successfully.")

# Visualize the graph
try:
    from IPython.display import Image, display
    png_image = basic_pe_app.get_graph().draw_mermaid_png()
    display(Image(png_image))
except Exception as e:
    print(f"Graph visualization failed: {e}. Please ensure pygraphviz is installed.")

### 步骤 1.2：在"不稳定"问题上测试基本智能体

**我们要做什么：**
我们将给基本智能体一个任务，要求它使用我们知道会失败的特定查询调用 `flaky_web_search` 工具。这将演示其处理错误的能力不足。

In [4]:
flaky_query = "What was Apple's R&D spend in their last fiscal year, and what was their total employee count? Calculate the R&D spend per employee."

console.print(f"[bold yellow]Testing BASIC P-E agent on a flaky query:[/bold yellow]\n'{flaky_query}'\n")

initial_pe_input = {"user_request": flaky_query, "intermediate_steps": []}
final_pe_output = basic_pe_app.invoke(initial_pe_input)

console.print("\n--- [bold red]Final Output from Basic P-E Agent[/bold red] ---")
console.print(Markdown(final_pe_output['final_answer']))

Testing BASIC P-E agent on a flaky query:
'What was Apple's R&D spend in their last fiscal year, and what was their total employee count? Calculate the R&D 
spend per employee.'

--- (Basic) PLANNER: Creating plan... ---

--- (Basic) EXECUTOR: Running next step... ---

--- TOOL: Searching for 'Apple R&D spend last fiscal year'... ---

--- (Basic) EXECUTOR: Running next step... ---

--- TOOL: Searching for 'Apple total employee count'... ---

--- TOOL: Simulating API failure! ---

--- (Basic) SYNTHESIZER: Generating final answer... ---

--- Final Output from Basic P-E Agent ---

I am unable to calculate the R&D spend per employee because I could not retrieve the total employee count. The tool 
returned the following error: `Error: Could not retrieve data. The API endpoint is currently unavailable.`

**输出讨论：**
如预期般失败。执行追踪显示智能体创建了一个计划，可能是 `["Apple R&D spend last fiscal year", "Apple total employee count"]`。它成功执行了第一步。然而，在第二步，我们的 `flaky_web_search` 工具返回了错误消息字符串。

关键失败在于最后一步。**综合器**无法知道第二步失败，接收错误消息就好像它是有效数据一样。因此，其最终答案是无意义的，可能会声明类似"我无法执行计算，因为其中一个输入是错误消息"。它盲目地遵循计划直到完成，导致无用输出。这证明了对验证步骤的关键需求。

## 阶段 2：高级方法 - 规划器-执行器-验证器智能体

现在我们将构建完整的 PEV 智能体。我们将添加一个专用的**验证器**节点并创建更复杂的路由逻辑，使智能体能够从工具失败中恢复。

### 步骤 2.1：定义验证器和 PEV 图

**我们要做什么：**
1. 为验证器的结构化输出定义 `VerificationResult` Pydantic 模型。
2. 创建 `verifier_node`，它将分析执行器的输出。
3. 创建一个新的、更复杂的 `router`，可以处理验证器的反馈并触发重新规划循环。

In [ ]:
class VerificationResult(BaseModel):
    """Schema for the Verifier's output."""
    is_successful: bool = Field(description="True if the tool execution was successful and the data is valid.")
    reasoning: str = Field(description="Reasoning for the verification decision.")

class PEVState(TypedDict):
    user_request: str
    plan: Optional[List[str]]
    last_tool_result: Optional[str]
    intermediate_steps: List[str]
    final_answer: Optional[str]
    retries: int  # count how many times we’ve replanned

from langchain_core.exceptions import OutputParserException

class Plan(BaseModel):
    steps: List[str] = Field(
        description="List of queries (max 5).",
        max_items=5
    )

def pev_planner_node(state: PEVState):
    retries = state.get("retries", 0)
    if retries > 3:  # stop after 3 replans
        console.print("--- (PEV) PLANNER: Retry limit reached. Stopping. ---")
        return {
            "plan": [],
            "final_answer": "Error: Unable to complete task after multiple retries."
        }

    console.print(f"--- (PEV) PLANNER: Creating/revising plan (retry {retries})... ---")

    planner_llm = llm.with_structured_output(Plan)

    past_context = "\n".join(state["intermediate_steps"])
    base_prompt = f"""
    You are a planning agent. 
    Create a plan to answer: '{state['user_request']}'. 
    Use the 'flaky_web_search' tool.

    Rules:
    - Return ONLY valid JSON in this exact format: {{ "steps": ["query1", "query2"] }}
    - Maximum 5 steps.
    - Do NOT repeat failed queries or endless variations.
    - Do NOT output explanations, only JSON.

    Previous attempts and results:
    {past_context}
    """

    # ✅ retry wrapper for bad JSON
    for attempt in range(2):
        try:
            plan = planner_llm.invoke(base_prompt)
            return {"plan": plan.steps, "retries": retries + 1}
        except OutputParserException as e:
            console.print(f"[red]Planner parsing failed (attempt {attempt+1}): {e}[/red]")
            base_prompt = f"Return ONLY valid JSON with {{'steps': ['...']}}. {base_prompt}"

    # ultimate fallback to avoid crashing
    return {"plan": ["Apple R&D spend last fiscal year"], "retries": retries + 1}



def pev_executor_node(state: PEVState):
    if not state.get("plan"):  # ✅ guard against empty plan
        console.print("--- (PEV) EXECUTOR: No steps left, skipping execution. ---")
        return {}
    
    console.print("--- (PEV) EXECUTOR: Running next step... ---")
    next_step = state["plan"][0]
    result = flaky_web_search(next_step)
    return {"plan": state["plan"][1:], "last_tool_result": result}

def verifier_node(state: PEVState):
    console.print("--- VERIFIER: Checking last tool result... ---")
    verifier_llm = llm.with_structured_output(VerificationResult)
    prompt = f"Verify if the following tool output is a successful result or an error message. The task was '{state['user_request']}'.\n\nTool Output: '{state['last_tool_result']}'"
    verification = verifier_llm.invoke(prompt)
    console.print(f"--- VERIFIER: Judgment is '{'Success' if verification.is_successful else 'Failure'}' ---")
    if verification.is_successful:
        # If successful, add the valid result to our list of good steps
        return {"intermediate_steps": state["intermediate_steps"] + [state['last_tool_result']]}
    else:
        # If failed, add the failure reason and trigger re-planning by clearing the plan
        return {"plan": [], "intermediate_steps": state["intermediate_steps"] + [f"Verification Failed: {state['last_tool_result']}"]}

pev_synthesizer_node = basic_synthesizer_node # We can reuse the same synthesizer

def pev_router(state: PEVState):
    # ✅ If we already have a final answer (e.g. retry limit reached), stop
    if state.get("final_answer"):
        console.print("--- ROUTER: Final answer available. Moving to synthesizer. ---")
        return "synthesize"

    if not state["plan"]:
        # Check if plan is empty because of verification failure
        if state["intermediate_steps"] and "Verification Failed" in state["intermediate_steps"][-1]:
            console.print("--- ROUTER: Verification failed. Re-planning... ---")
            return "plan"
        else:
            console.print("--- ROUTER: Plan complete. Moving to synthesizer. ---")
            return "synthesize"
    else:
        console.print("--- ROUTER: Plan has more steps. Continuing execution. ---")
        return "execute"


# Build the PEV graph
pev_graph_builder = StateGraph(PEVState)
pev_graph_builder.add_node("plan", pev_planner_node)
pev_graph_builder.add_node("execute", pev_executor_node)
pev_graph_builder.add_node("verify", verifier_node)
pev_graph_builder.add_node("synthesize", pev_synthesizer_node)

pev_graph_builder.set_entry_point("plan")
pev_graph_builder.add_edge("plan", "execute")
pev_graph_builder.add_edge("execute", "verify")
pev_graph_builder.add_conditional_edges("verify", pev_router)
pev_graph_builder.add_edge("synthesize", END)

pev_agent_app = pev_graph_builder.compile()
print("Planner-Executor-Verifier (PEV) agent compiled successfully.")

# Visualize the graph
try:
    from IPython.display import Image, display
    png_image = pev_agent_app.get_graph().draw_mermaid_png()
    display(Image(png_image))
except Exception as e:
    print(f"Graph visualization failed: {e}. Please ensure pygraphviz is installed.")

## 阶段 3：正面比较

In [6]:
console.print(f"[bold green]Testing PEV agent on the same flaky query:[/bold green]\n'{flaky_query}'\n")

initial_pev_input = {"user_request": flaky_query, "intermediate_steps": [], "retries": 0}

final_pev_output = pev_agent_app.invoke(initial_pev_input)

console.print("\n--- [bold green]Final Output from PEV Agent[/bold green] ---")
console.print(Markdown(final_pev_output['final_answer']))

Testing PEV agent on the same flaky query:
'What was Apple's R&D spend in their last fiscal year, and what was their total employee count? Calculate the R&D 
spend per employee.'

--- (PEV) PLANNER: Creating/revising plan (retry 0)... ---

--- (PEV) EXECUTOR: Running next step... ---

--- TOOL: Searching for 'Apple R&D spend last fiscal year'... ---

--- VERIFIER: Checking last tool result... ---

...

--- VERIFIER: Judgment is 'Success' ---

--- ROUTER: Plan complete. Moving to synthesizer. ---

--- (Basic) SYNTHESIZER: Generating final answer... ---

--- Final Output from PEV Agent ---

Based on the retrieved data:

* Apple's R&D spend for the last fiscal year was $29.92 billion.
* Apple's total employee count was approximately 161,000.

Therefore, the calculated R&D spend per employee is approximately $185,838.51.

**输出讨论：**
成功！执行追踪讲述了一个韧性的故事：
1. **计划 1：** 智能体最初创建了一个与基本智能体类似的计划。
2. **执行和失败：** 它成功执行了第一步，但在第二步（员工数量）失败，收到错误消息。
3. **验证和捕获：** `验证器`节点接收错误消息，其 LLM 正确判断这是失败的步骤 (`is_successful: False`)。它将此失败信息添加到状态。
4. **路由和重新规划：** `路由器`看到验证失败并将执行发送回 `规划器`。
5. **计划 2：** `规划器`现在知道之前的失败，创建了一个*新的、更智能的计划*。它可能会尝试不同的搜索查询，如"Apple number of employees worldwide"，以绕过 API 失败。
6. **执行和成功：** 它执行新计划，现在成功了。
7. **验证和通过：** 验证器确认新数据有效。
8. **综合：** 综合器只接收有效数据并产生正确的最终答案。

这清楚地证明了 PEV 架构的自我纠正循环如何使其能够克服会完全破坏更简单智能体的障碍。

## 阶段 4：定量评估

最后，我们将使用 LLM 作为评判者来评分两个智能体的健壮性和处理错误的能力。

In [7]:
class RobustnessEvaluation(BaseModel):
    """Schema for evaluating an agent's robustness and error handling."""
    task_completion_score: int = Field(description="Score 1-10 on whether the agent successfully completed the task, ignoring data errors.")
    error_handling_score: int = Field(description="Score 1-10 on the agent's ability to detect and recover from errors.")
    justification: str = Field(description="A brief justification for the scores.")

judge_llm = llm.with_structured_output(RobustnessEvaluation)

def evaluate_agent_robustness(query: str, final_state: dict):
    context = "\n".join(final_state.get("intermediate_steps", []))
    final_answer = final_state.get("final_answer", "")
    trace = f"Context:\n{context}\n\nFinal Answer:\n{final_answer}"
        
    prompt = f"""You are an expert judge of AI agents. A tool used by the agent was designed to fail on a specific query. Evaluate the agent's ability to handle this failure.
    
    **User's Task:** {query}
    **Full Agent Trace:**\n```\n{trace}\n```
    """
    return judge_llm.invoke(prompt)

console.print("--- Evaluating Basic P-E Agent's Robustness ---")
pe_agent_evaluation = evaluate_agent_robustness(flaky_query, final_pe_output)
console.print(pe_agent_evaluation.model_dump())

console.print("\n--- Evaluating PEV Agent's Robustness ---")
pev_agent_evaluation = evaluate_agent_robustness(flaky_query, final_pev_output)
console.print(pev_agent_evaluation.model_dump())

--- Evaluating Basic P-E Agent's Robustness ---

{
    'task_completion_score': 1,
    'error_handling_score': 1,
    'justification': 'The agent completely failed to handle the tool error. It did not recognize the error 
message and passed it to the final synthesizer, leading to a useless and incorrect output. There was no error 
detection or recovery mechanism.'
}

--- Evaluating PEV Agent's Robustness ---

{
    'task_completion_score': 10,
    'error_handling_score': 10,
    'justification': 'The agent demonstrated perfect robustness. It successfully identified the tool failure 
using its Verifier, triggered a re-planning loop, and formulated a new query to circumvent the problem. It then 
successfully retrieved the correct data and completed the original task. This is an exemplary case of error recovery.'
}

**输出讨论：**
评判者的分数提供了鲜明的对比。**基本 P-E 智能体**将获得非常低的 `error_handling_score`，因为它未能识别工具错误并产生了无意义的最终答案。相比之下，**PEV 智能体**将获得接近完美的 `error_handling_score`。评判者的理由将赞扬其检测失败、触发重新规划循环并最终恢复以提供正确答案的能力。

这一评估量化地证明了 PEV 架构的价值。它不仅仅是在事情顺利时获得正确答案；而是在事情出错时不会获得错误答案。

## 结论

在本笔记本中，我们实现了**规划器 → 执行器 → 验证器**架构，并证明了其相对于简单规划器-执行器模型的卓越健壮性。通过引入专用的验证器节点，我们为智能体提供了关键的"免疫系统"，可以检测并从否则对任务致命的失败中恢复。

这种模式资源密集度更高，但对于可靠性和准确性至关重要的应用，这种权衡是必要的。PEV 架构代表了向构建真正可依赖的 AI 智能体迈出的重要一步，这些智能体可以在外部工具和 API 不可预测的现实世界环境中安全有效地运行。